In [11]:
import numpy as np
import os

def cal_suc(task, env):
    if env == 'simulator':
        path = '/mnt/sda3/muyanglyu/results/'
        os.chdir(path)
        target = 'simulator'
    elif env == 'model':
        path = '/home/xiaoliu/work/content_structure/vp2/'
        os.chdir(path)
        target = 'content_structure_interface_woft_epoch33'
    
    if task == 'robosuite':
        path += f'vp2/outputs/rs_case_study/g/{target}/'
    else:
        path += f'vp2/outputs/rd_case_study/g/{target},{task}/'

    seeds = [2, 3, 4]
    success_rates = []
    for seed in seeds:
        file_path = path + f'{seed}/' + 'all_rews.txt'
        data = np.loadtxt(file_path, dtype=float, delimiter=',')
        data = (data[:, 1] + (not task == 'robosuite')) < 0.05
        success_rates.append(np.mean(data))
    success_rates = np.array(success_rates)

    return success_rates



In [9]:
# ['push_red', 'push_blue', 'push_green', 'flat_block_off_table', 'upright_block_off_table', 'open_slide', 'open_drawer']
# ['robosuite']

tasks = ['push_red', 'push_blue', 'push_green', 'flat_block_off_table', 'upright_block_off_table', 'open_drawer','robosuite', 'open_slide']
# tasks = ['robosuite']
suc = []


In [3]:
for task in tasks:
    model_suc = cal_suc(task, 'model')
    simulator_suc = cal_suc(task, 'simulator')
    nor_suc = np.mean(model_suc) / np.mean(simulator_suc)
    suc.append(nor_suc)
suc = np.array(suc)
print(f'aggregate: {round(np.mean(suc), 4)}')

FileNotFoundError: /mnt/sda3/muyanglyu/results/vp2/outputs/rd_case_study/g/simulator,push_red/2/all_rews.txt not found.

In [12]:
for task in tasks:
    m_auc = np.mean(cal_suc(task, 'model'))
    s_auc = np.std(cal_suc(task, 'model'))
    print(f"task {task} success rate is {np.round(m_auc * 100, 2)} ± {np.round(s_auc * 100, 2) } %")
    


task push_red success rate is 0.0 ± 0.0 %
task push_blue success rate is 51.11 ± 5.67 %
task push_green success rate is 11.11 ± 1.57 %
task flat_block_off_table success rate is 0.0 ± 0.0 %
task upright_block_off_table success rate is 44.44 ± 3.14 %
task open_drawer success rate is 6.67 ± 4.71 %
task robosuite success rate is 43.33 ± 1.25 %
task open_slide success rate is 0.0 ± 0.0 %


In [10]:
s_auc

0.04330127018922193

In [41]:
import base64
import os

failed_cards = []
success_cards = []
for idx in range(len(data)):
    gif_path = path + f'traj_{idx}/traj_{idx}_vis.gif'
    with open(gif_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
    img_src = f"data:image/gif;base64,{encoded_string}"
    card = f'''
    <div class="card">
        <div class="title">Traj {idx}</div>
        <img src="{img_src}" alt="Traj {idx}">
    </div>
    '''
    if not data[idx]:
        failed_cards.append(card)
    else:
        success_cards.append(card)

if task == 'robosuite':
    cards_file = 'vp2/outputs_gif/robosuite/'
else:
    cards_file = f'vp2/outputs_gif/robodesk/{task}/'
if not os.path.exists(cards_file):
    os.makedirs(cards_file)


In [42]:

def generate_html(cards, type_, output_file):
    full_html = f'''
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>案例汇总</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                background-color: #f4f4f9;
                padding: 20px;
            }}
            h1 {{ text-align: center; color: #333; }}
            .container {{
                display: flex;
                flex-wrap: wrap;
                justify-content: center;
                gap: 20px;
            }}
            .card {{
                background: white;
                padding: 10px;
                border-radius: 8px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                text-align: center;
                width: 300px; /* 卡片宽度 */
            }}
            .card img {{
                max-width: 100%;
                height: auto;
                border: 1px solid #ddd;
            }}
            .title {{
                font-weight: bold;
                margin-bottom: 10px;
                color: #d9534f;
            }}
        </style>
    </head>
    <body>
        <h1>共有 {len(cards)} 个{type_}案例</h1>
        <div class="container">
            {''.join(cards)}
        </div>
    </body>
    </html>
    '''
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(full_html)

In [43]:
generate_html(failed_cards, '失败', os.path.join(cards_file, 'vis_failed_cases.html'))
generate_html(success_cards, '成功', os.path.join(cards_file, 'vis_successful_cases.html'))

In [3]:
import numpy as np

np.linspace(0, 1, 5 + 1).shape

(6,)